In [2]:
import neo4j
import pandas as pd
from IPython.display import display
import numpy as np
import requests
from io import StringIO

### Refugee Movement Around the World Analysis 

There is a United Nations agency that track refugee movement.

Dataset: https://github.com/rfordatascience/tidytuesday/blob/master/data/2023/2023-08-22/readme.md

The dataset gives a list of countries, and for each pair of countries, how many people emigrate from one country and immigrate to another country.

#### Suggestions for creating 2 different graphs:

- Create a node for each country, with a one way relationship from the country the refugees are emigrating from to the country the refugees are immigrating to, with the weight of the relationship the number of refugees
- Create a node for each country, with a one way relationship from the country the refugees are immigrating toto the country the refugees are emigrating from, with the weight of the relationship the number of refugees

#### Suggestions for data science graph algorithms and what insights they might help to inform us of:

- For the first graph, PageRank would tell you who the most influential countries in granting asylum
- For the second graph, PageRank would tell you who the most influential countries in creating refugees
- Select the top X asylum granters, and for each top asylum granter, run a Minimum Spanning Tree on the second graph to see how refugees flow from the top refugee creators
- Select the top X refugee creators, and for each top refugee creator, run a Minimum Spanning Tree on the first graph to see how refugees flow to the top asylum granters
- On either graph, run Betweenness Centrality, with a high betweenness showing the intermediate counties that act as places of temporary refuge in the flow of refugees
- On the first graph, Shortest Path between two countries would show the most direct flow of refugees between them
- On the second graph, repeat showing the most direct flow of refugees the other way
- On the either graph, run Degree Centrality, with a high degree centrality showing the countries with the most creation of refugees / grant asylum
- On either graph, run Harmonic Centrality, with a high harmonic centrality showing the intermediate countries with the highest volume of refugees
- On either graph, run Louvain Modularity with communities that physically span the sea showing sea port routes that refugees might be taking

_You may also want to consider making graphs for different time periods to analyze if the top asylum granting countries, top refugee creating countries, top intermediate countries, top paths, top sea routes, etc. have changed over the years._

In [3]:
url = "https://raw.githubusercontent.com/rfordatascience/tidytuesday/main/data/2023/2023-08-22/population.csv"
response = requests.get(url, verify=False)
population = pd.read_csv(StringIO(response.text))
print("", population.shape)
population.head()

/opt/conda/lib/python3.9/site-packages/urllib3/connectionpool.py:1013: InsecureRequestWarning: Unverified HTTPS request is being made to host 'raw.githubusercontent.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/1.26.x/advanced-usage.html#ssl-warnings
  warnings.warn(


 (64809, 16)


,year,coo_name,coo,coo_iso,coa_name,coa,coa_iso,refugees,asylum_seekers,returned_refugees,idps,returned_idps,stateless,ooc,oip,hst
0,2010,Afghanistan,AFG,AFG,Afghanistan,AFG,AFG,0,0,0,351907,3366,0,838250,NaN,NaN
1,2010,Iran (Islamic Rep. of),IRN,IRN,Afghanistan,AFG,AFG,30,21,0,0,0,0,0,NaN,NaN
2,2010,Iraq,IRQ,IRQ,Afghanistan,AFG,AFG,6,0,0,0,0,0,0,NaN,NaN
3,2010,Pakistan,PAK,PAK,Afghanistan,AFG,AFG,6398,9,0,0,0,0,0,NaN,NaN
4,2010,Egypt,ARE,EGY,Albania,ALB,ALB,5,0,0,0,0,0,0,NaN,NaN


## Web server interface at https://xxxx:7473

#### Update - since the videos were filmed, neo4j requires a longer, more complex password, so the newest password is here:

**Username: neo4j**

**Password: ucb_mids_w205**

The above web server allows and interactive GUI which can output graphs visually in addition to table like output.  The nodes in the graphs can be moved around with the mouse to make the graphs more readable.


#### Basics:

```:server connect``` - connect to the server, username is "neo4j", password is "ucb_mids_w205"


```:server status``` - shows that username and server you are logged into


```:clear``` - clears off old cells


```show databases``` - note that community edition only has 1 application database that we can use neo4j, we cannot create now use other databases, we have to wipe out neo4j database for each new graph


## Query to get full Graph:

```match (n) return n```

## Connect, login, create driver, create session; with community edition, we can only use 1 database, the "neo4j" database

In [4]:
driver = neo4j.GraphDatabase.driver(uri="neo4j://neo4j:7687", auth=("neo4j","ucb_mids_w205"))

In [5]:
session = driver.session(database="neo4j")

## my_neo4j_wipe_out_database() - since community edition can only have 1 database "neo4j", this function will wipe out all the nodes and relationships

In [6]:
def my_neo4j_wipe_out_database():
    "wipe out database by deleting all nodes and relationships"
    
    query = "match (node)-[relationship]->() delete node, relationship"
    session.run(query)
    
    query = "match (node) delete node"
    session.run(query)

## my_neo4j_run_query_pandas() will run a Cypher query and put the results in a Pandas dataframe; easy to see how you can use Python to manipulate the returned data

In [7]:
def my_neo4j_run_query_pandas(query, **kwargs):
    "run a query and return the results in a pandas dataframe"
    
    result = session.run(query, **kwargs)
    
    df = pd.DataFrame([r.values() for r in result], columns=result.keys())
    
    return df

## my_neo4j_nodes_relationships() will print the nodes (assumes a name property) and relationships

In [8]:
def my_neo4j_nodes_relationships():
    "print all the nodes and relationships"
   
    print("-------------------------")
    print("  Nodes:")
    print("-------------------------")
    
    query = """
        match (n) 
        return n.name as node_name, labels(n) as labels
        order by n.name
    """
    
    df = my_neo4j_run_query_pandas(query)
    
    number_nodes = df.shape[0]
    
    display(df)
    
    print("-------------------------")
    print("  Relationships:")
    print("-------------------------")
    
    query = """
        match (n1)-[r]->(n2) 
        return n1.name as node_name_1, labels(n1) as node_1_labels, 
            type(r) as relationship_type, n2.name as node_name_2, labels(n2) as node_2_labels
        order by node_name_1, node_name_2
    """
    
    df = my_neo4j_run_query_pandas(query)
    
    number_relationships = df.shape[0]
    
    display(df)
    
    density = (2 * number_relationships) / (number_nodes * (number_nodes - 1))
    
    print("-------------------------")
    print("  Density:", f'{density:.1f}')
    print("-------------------------")
    

## Creating Database for Refugee Data

In [9]:
my_neo4j_wipe_out_database()

aggregated = population.groupby(
    ["coo_name", "coo_iso", "coa_name", "coa_iso"],
    as_index=False
)["refugees"].sum()

# Remove self-loops with 0 weight before loading
aggregated = aggregated[~((aggregated["coo_iso"] == aggregated["coa_iso"]) & (aggregated["refugees"] == 0))]

# Remove all 0 weight relationships
aggregated = aggregated[aggregated["refugees"] > 0]

origin_countries = aggregated[["coo_name", "coo_iso"]].rename(
    columns={"coo_name": "name", "coo_iso": "iso"}
)
asylum_countries = aggregated[["coa_name", "coa_iso"]].rename(
    columns={"coa_name": "name", "coa_iso": "iso"}
)
all_countries = pd.concat([origin_countries, asylum_countries]).drop_duplicates(subset="iso")

for _, row in all_countries.iterrows():
    query = """
        MERGE (c:Country {iso: $iso})
        SET c.name = $name
    """
    session.run(query, iso=row["iso"], name=row["name"])

for _, row in aggregated.iterrows():
    query = """
        MATCH (origin:Country {iso: $coo_iso})
        MATCH (asylum:Country {iso: $coa_iso})
        MERGE (origin)-[r:DISPLACED_TO]->(asylum)
        SET r.weight = $weight
    """
    session.run(query,
                coo_iso=row["coo_iso"],
                coa_iso=row["coa_iso"],
                weight=int(row["refugees"]))

print("Done")

Done


### Checking for Self-Loops and making sure they are only added if not 0

In [10]:
#Self-loop countries with weight
query = """
MATCH (a:Country)-[r:DISPLACED_TO]->(a)
RETURN a.name AS country, r.weight AS refugees
"""

my_neo4j_run_query_pandas(query)

,country,refugees
0,Australia,5
1,Netherlands (Kingdom of the),5
2,United Kingdom of Great Britain and Northern I...,58
3,United Rep. of Tanzania,11


(_self-loop countries are sending refugees to other countries and they get returned?_)

### Finding all islands/disconnected nodes

In [11]:
query = """
MATCH (c:Country)
WHERE NOT (c)-[:DISPLACED_TO]->() AND NOT ()-[:DISPLACED_TO]->(c)
RETURN c.name AS country, c.iso AS iso
ORDER BY country
"""
my_neo4j_run_query_pandas(query)

,country,iso


# Graph Algorithms

In [12]:
# Query Call to intialize graph (just in case)
query = "CALL gds.graph.drop('refugee_graph', false) yield graphName"
session.run(query)

query = """
CALL gds.graph.project(
    'refugee_graph',
    'Country',
    'DISPLACED_TO',
    {relationshipProperties: 'weight'}
)
"""
session.run(query)

### Degree Centrality

\# of connections to a country (weighted with \# of refugees from the connection) 

_higher score = more refugees movement_

In [13]:
# Degree Centrality
query = """
CALL gds.degree.stream('refugee_graph', {relationshipWeightProperty: 'weight'})
YIELD nodeId, score
RETURN gds.util.asNode(nodeId).name AS country, score AS degree_centrality
ORDER BY degree_centrality DESC
"""
my_neo4j_run_query_pandas(query)

,country,degree_centrality
0,Syrian Arab Rep.,57200667.0
1,Afghanistan,37628196.0
2,South Sudan,16839631.0
3,Somalia,12564708.0
4,Myanmar,10058729.0
...,...,...
200,Anguilla,5.0
201,Palau,5.0
202,Vanuatu,5.0
203,Sint Maarten (Dutch part),0.0


### Harmonic Centrality
How close a country is to other countries (goes through all direct paths)  

_higher score = many different countries send immigrants to the countries_

In [14]:
# Harmonic Centrality
query = """
CALL gds.closeness.harmonic.stream('refugee_graph')
YIELD nodeId, score
RETURN gds.util.asNode(nodeId).name AS country, score AS harmonic_centrality
ORDER BY harmonic_centrality DESC
"""
my_neo4j_run_query_pandas(query)

,country,harmonic_centrality
0,Canada,0.936275
1,United States of America,0.936275
2,Germany,0.899510
3,United Kingdom of Great Britain and Northern I...,0.851307
4,France,0.835784
...,...,...
200,Tibetan,0.000000
201,Timor-Leste,0.000000
202,Tonga,0.000000
203,Unknown,0.000000


### Louvain Modularity
Groups countries by large amounts of flow between grouped countries  

_most groups will be geographically closer while some outliers just have a lot of sea traffic between the countries_

In [24]:
# Louvain Modularity
query = """
CALL gds.louvain.stream('refugee_graph', {
    relationshipWeightProperty: 'weight',
    includeIntermediateCommunities: true,
    tolerance: 0.5
})
YIELD nodeId, communityId, intermediateCommunityIds
RETURN gds.util.asNode(nodeId).name AS country, 
       communityId AS community, 
       intermediateCommunityIds AS intermediate_community
ORDER BY community, country ASC
"""
my_neo4j_run_query_pandas(query)

,country,community,intermediate_community
0,Afghanistan,0,[0]
1,Pakistan,0,[0]
2,Fiji,70,[70]
3,Indonesia,70,[70]
4,Nauru,70,[70]
...,...,...,...
200,Yemen,192,[192]
201,Zambia,192,[192]
202,Zimbabwe,192,[192]
203,Sint Maarten (Dutch part),203,[203]


In [22]:
# Try these one at a time
tolerances = [0.001, 0.01, 0.1, 0.5, 1]

for t in tolerances:
    query = """
    CALL gds.louvain.stream('refugee_graph', {
        relationshipWeightProperty: 'weight',
        includeIntermediateCommunities: true,
        tolerance: $tolerance
    })
    YIELD nodeId, communityId
    RETURN communityId AS community,
           count(nodeId) AS num_countries,
           collect(gds.util.asNode(nodeId).name) AS countries
    ORDER BY num_countries DESC
    """
    print(f"\n=== Tolerance: {t} ===")
    df = my_neo4j_run_query_pandas(query, tolerance=t)
    print(f"Number of communities: {len(df)}")
    display(df[["community", "num_countries"]].head(10))


=== Tolerance: 0.001 ===
Number of communities: 6


,community,num_countries
0,192,186
1,171,8
2,111,7
3,0,2
4,203,1
5,204,1



=== Tolerance: 0.01 ===
Number of communities: 6


,community,num_countries
0,192,186
1,171,8
2,111,7
3,0,2
4,203,1
5,204,1



=== Tolerance: 0.1 ===
Number of communities: 6


,community,num_countries
0,192,186
1,171,8
2,111,7
3,0,2
4,203,1
5,204,1



=== Tolerance: 0.5 ===
Number of communities: 8


,community,num_countries
0,192,180
1,70,8
2,111,7
3,171,5
4,0,2
5,190,1
6,203,1
7,204,1



=== Tolerance: 1 ===
Number of communities: 8


,community,num_countries
0,192,180
1,70,8
2,111,7
3,171,5
4,0,2
5,190,1
6,203,1
7,204,1
